In [1]:
import os
import sys
import dask.dataframe as dd

module_path = os.path.abspath(os.path.join("."))
if module_path not in sys.path:
    sys.path.append(module_path)

from utilities import walk_through_files, save_df_in_chunks

from dask.distributed import Client

client = Client()  # Connect to distributed cluster and override default
client

RECORDS_PATH = os.path.join("raw-data", "sg_subzones_devices")
SAVEPATH = os.path.join("intermediate-files", "tt-appearances-areadate")

D:\ProgramData\Anaconda3\lib\site-packages\distributed\node.py:155: UserWarning: Port 8787 is already in use.
Perhaps you already have a cluster running?
Hosting the HTTP server on port 54067 instead
  http_address["port"], self.http_server.port


Client Scheduler: tcp://127.0.0.1:54070 Dashboard: http://127.0.0.1:54067/status,Cluster Workers: 4 Cores: 12 Memory: 34.29 GB


In [3]:
file_lists = tuple(walk_through_files(RECORDS_PATH, "csv.gz", fullpath=True))
len(file_lists)

In [4]:
%%time
ddf = (
    (
        dd.read_csv(
            file_lists,
            compression="gzip",
            usecols=[0],
            names=["deviceID"],
            blocksize=None,
        )
        .groupby("deviceID")
        .size()
        .reset_index()
        .rename(columns={0: "tt_appearances_areadate"})
        .astype({"tt_appearances_areadate": "int"})
    )
    .compute()
    .sort_values("tt_appearances_areadate", ascending=False, ignore_index=True)
)

Wall time: 7min 39s


In [ ]:
save_df_in_chunks(ddf, SAVEPATH, 100, "device-appearances-chunk", "csv.gz", "gzip")

In [5]:
len(ddf[ddf["tt_appearances_areadate"] == 1])

1000570

In [6]:
len(ddf[ddf["tt_appearances_areadate"] > 400])

1667

In [7]:
len(ddf[ddf["tt_appearances_areadate"] < 50])

17304335

In [8]:
bins = [x / 100 for x in list(range(5, 100 + 5, 5))]

ddf.describe(percentiles=bins)

,tt_appearances_areadate
count,1.749219e+07
mean,6.366656e+00
std,1.380609e+01
min,1.000000e+00
5%,1.000000e+00
10%,2.000000e+00
15%,2.000000e+00
20%,2.000000e+00
25%,2.000000e+00
30%,2.000000e+00
